# Example 11.6 / 1: Declare Complementarity Conditions

Companion notebook for **11.6 Complementarity Constraints** in the AMPL lab Documentation, section "Declare Complementarity Conditions". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Declare Complementarity Conditions

At the lower bound the residual has one sign, at the upper bound the opposite sign, and in the interior the residual is zero.

The bounded form with $0\le x\le10$ and residual $F(x)=x-c$ reaches each case as `c` changes.
HiGHS solves it through the MP conversion of `complements` into logical constraints, which needs
finite bounds on the variables involved:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param c;  # shifts the residual x - c

var x >= 0, <= 10;  # bounds also appear in the pair

subject to BoundedPair:  # x at a bound or residual zero
    0 <= x <= 10 complements x - c;

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for value in (-1, 4, 12):
    ampl.param["c"] = value
    ampl.solve(solver="highs", verbose=False)
    x = ampl.var["x"].value()
    print("c =", value, ampl.solve_result, "x =", round(x, 6), "F(x) =", round(x - value, 6))

c = -1 solved x = 0.0 F(x) = 1.0
c = 4 solved x = 4.0 F(x) = 0.0
c = 12 solved x = 10.0 F(x) = -2.0
